# <font color="#76b900">**Notebook 0:** Deploying a NIM on DGX Spark</font>

**This notebook is specific to running the workshop on a DGX Spark.**

> **Fresh machine?** Run `./setup-dgx-spark.sh` in a terminal first. It installs
> JupyterLab and the `llm-eval-workshop` kernel and performs these checks automatically.
> This notebook is the expanded, interactive NIM walkthrough.

On a DGX Spark (GB10 Grace-Blackwell, `aarch64`, 128 GB unified memory)
there is no pre-baked service — you deploy the NIM yourself, locally, on the box.

In this notebook you will:
- Verify the DGX Spark environment (GPU, Docker, GPU-in-container support).
- Authenticate to the NVIDIA NGC container registry.
- Launch the **NVIDIA Nemotron Nano 9B v2** NIM as a local container.
- Wait for it to become healthy and send a first test query.

When it finishes, the NIM is reachable at **`http://localhost:8000`** and serves
the model id **`nvidia/nemotron-nano-9b-v2`** (this is the id the local NIM
reports at `/v1/models`; note it has a single `nvidia` prefix, unlike the public
catalog name `nvidia/nvidia-nemotron-nano-9b-v2`). You can then proceed to
`01-NIM-Evaluation.ipynb`.

> A companion step-by-step reference lives in
> [`guide/DGX-Spark-Setup.md`](guide/DGX-Spark-Setup.md).

<br><hr>

## **Step 1 — Verify the environment**

Confirm the GB10 GPU is visible, Docker is installed, and the NVIDIA container runtime is available:

In [ ]:
!echo '=== GPU ==='          && nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo 'nvidia-smi not found'
!echo '=== Docker ==='       && docker --version || echo 'docker not found'
!echo '=== GPU-in-Docker (CDI) ===' && (nvidia-ctk cdi list 2>/dev/null | grep -q 'nvidia.com/gpu=all' && echo 'CDI device nvidia.com/gpu=all available' || echo 'CDI not found — see note below')
!echo '=== Docker runtimes ===' && (docker info 2>/dev/null | grep -i 'Runtimes')
!echo '=== Arch ==='         && uname -m

You should see the **`NVIDIA GB10`** GPU, a Docker version, architecture
`aarch64`, and confirmation that GPUs are usable inside containers.

**How GPUs reach the container:** this notebook uses **CDI** (the Container
Device Interface) via `--device nvidia.com/gpu=all`, which is how the NVIDIA
Container Toolkit exposes GPUs on DGX Spark. If the CDI check above passed,
you're ready.

- If CDI is **not** found, generate the spec once:
  ```bash
  sudo nvidia-ctk cdi generate --output=/etc/cdi/nvidia.yaml
  ```
- Alternatively, if your Docker has the legacy **`nvidia` runtime** registered
  (visible in the *Runtimes* line above), you can instead launch with
  `--runtime=nvidia --gpus all`. Configure it with:
  ```bash
  sudo nvidia-ctk runtime configure --runtime=docker && sudo systemctl restart docker
  ```

<br><hr>

## **Step 2 — Provide your NGC API key**

NIM containers are pulled from NVIDIA's registry (`nvcr.io`), which requires an
**NGC API key**. Create one at <https://build.nvidia.com> (Account → **API Keys**)
— it looks like `nvapi-...`.

Run the next cell and paste the key when prompted. It is stored only in this
kernel's environment (via `getpass`, so it won't be echoed or saved in the
notebook).

In [ ]:
import os, getpass

if not os.environ.get("NGC_API_KEY"):
    os.environ["NGC_API_KEY"] = getpass.getpass("Enter your NGC API key (nvapi-...): ")

assert os.environ["NGC_API_KEY"].startswith("nvapi-"), "That doesn't look like an NGC API key (should start with 'nvapi-')."
print("NGC_API_KEY is set.")

Log in to the NGC container registry. The username is the literal string `$oauthtoken` (not your email):

In [ ]:
!echo "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin

You should see `Login Succeeded`.

<br><hr>

## **Step 3 — Configure and launch the NIM**

We deploy the **Nemotron Nano 9B v2** NIM. The GB10's 128 GB of unified memory
runs it comfortably.

> **Use the `-dgx-spark` image on DGX Spark.** NVIDIA publishes a Spark-specific
> NIM build (`...-dgx-spark`) compiled for the GB10's `aarch64` CPU. If you ever need to check the exact current tag,
> open the model's **Deploy** tab at
> <https://build.nvidia.com/nvidia/nvidia-nemotron-nano-9b-v2/deploy> or the DGX
> Spark NIM playbook at <https://build.nvidia.com/spark/nim-llm>.

In [ ]:
import os

# --- Model / NIM selection ---------------------------------------------------
# `setdefault` means an exported environment variable wins, so you can switch
# model without editing this cell:
#
#   export NIM_IMAGE=nvcr.io/nim/meta/llama-3.1-8b-instruct:latest
#   export NIM_CONTAINER=llama-31-8b
#   export MODEL_ID=meta/llama-3.1-8b-instruct
#   export NIM_TOKENIZER=meta-llama/Llama-3.1-8B-Instruct   # used by notebook 01
#
# A NIM is a per-model *container image*, not a HuggingFace repo id — take the
# image and tag from the model's Deploy tab on build.nvidia.com. On DGX Spark
# prefer a `-dgx-spark` build (aarch64/GB10) where one is published; a generic
# x86 image will not run on the GB10.
os.environ.setdefault("NIM_IMAGE",       "nvcr.io/nim/nvidia/nvidia-nemotron-nano-9b-v2-dgx-spark:latest")
os.environ.setdefault("NIM_CONTAINER",   "nemotron-nano")
os.environ.setdefault("NIM_PORT",        "8000")
os.environ.setdefault("MODEL_ID",        "nvidia/nemotron-nano-9b-v2")
os.environ.setdefault("LOCAL_NIM_CACHE", os.path.expanduser("~/.cache/nim"))
# -----------------------------------------------------------------------------

os.makedirs(os.environ["LOCAL_NIM_CACHE"], exist_ok=True)
print("Image:    ", os.environ["NIM_IMAGE"])
print("Container:", os.environ["NIM_CONTAINER"])
print("Port:     ", os.environ["NIM_PORT"])
print("Model ID: ", os.environ["MODEL_ID"])
print("Cache:    ", os.environ["LOCAL_NIM_CACHE"])

### NIM Service Re-Check
Checks whether the container is already running or stopped before acting, so re-running it is always safe. On the **first run** the NIM downloads and optimises the model (several minutes); subsequent runs reuse the cache and are fast.

In [ ]:
%%bash
# Launch only if the container isn't already running or stopped — avoids
# destroying a live NIM when the cell is re-run.
if docker ps -q --filter "name=^${NIM_CONTAINER}$" | grep -q .; then
    echo "Container '$NIM_CONTAINER' is already running — nothing to do."
elif docker ps -aq --filter "name=^${NIM_CONTAINER}$" | grep -q .; then
    echo "Container '$NIM_CONTAINER' exists but is stopped — starting it."
    docker start "$NIM_CONTAINER"
else
    echo "Launching '$NIM_CONTAINER' from $NIM_IMAGE ..."
    docker run -d --name "$NIM_CONTAINER" \
        --device nvidia.com/gpu=all \
        --shm-size=16GB \
        -e NGC_API_KEY \
        -v "${LOCAL_NIM_CACHE}:/opt/nim/.cache" \
        -u $(id -u) \
        -p "${NIM_PORT}:8000" \
        "$NIM_IMAGE"
    echo "Container launched. Follow log progress in the next cell."
fi

Peek at the most recent log lines (re-run to refresh). Look for lines about the model loading and the API server starting:

In [ ]:
!docker logs --tail 25 "$NIM_CONTAINER" 2>&1 | tail -25

<br><hr>

## **Step 4 — Wait until the NIM is ready**

Poll the health endpoint until it returns HTTP 200. On the first run this may
take several minutes while the model downloads. The cell stops with a useful error
after 30 minutes instead of waiting forever.

In [ ]:
import os, time, requests
from IPython.display import clear_output

port = os.environ["NIM_PORT"]
url = f"http://localhost:{port}/v1/health/ready"
deadline = time.monotonic() + 1800

while time.monotonic() < deadline:
    try:
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            clear_output(wait=True)
            print("Service is up and running!")
            break
        message = f"status {r.status_code}"
    except requests.exceptions.RequestException as exc:
        message = str(exc)
    clear_output(wait=True)
    print("Waiting for service to be ready:", message)
    time.sleep(10)
else:
    raise TimeoutError(
        "NIM was not ready after 30 minutes. Run: "
        f"docker logs {os.environ['NIM_CONTAINER']}"
    )

<br><hr>

## **Step 5 — Verify the model and send a first query**

In [ ]:
model_response = requests.get(f"http://localhost:{port}/v1/models", timeout=10)
model_response.raise_for_status()
served_model = model_response.json()["data"][0]["id"]
print("served model id:", served_model)

# The id the NIM reports is authoritative — it often differs from the catalog
# name. Adopt it so the cells below (and notebooks 01/02) line up.
if served_model != os.environ["MODEL_ID"]:
    print(f"note: MODEL_ID was {os.environ['MODEL_ID']!r}; using the served id instead.")
    os.environ["MODEL_ID"] = served_model

The model id should be `nvidia/nemotron-nano-9b-v2`. Now a quick chat completion (we add a `/no_think` system message to get a concise answer — Nemotron Nano is a reasoning model and otherwise prepends its chain-of-thought):

In [ ]:
model_id = os.environ["MODEL_ID"]

# Nemotron Nano is a reasoning model and prepends its chain-of-thought unless
# told not to. Other models have no such directive, so send no system message.
system_prompt = os.environ.get(
    "NIM_SYSTEM_PROMPT", "/no_think" if "nemotron" in model_id.lower() else ""
)
messages = [{"role": "system", "content": system_prompt}] if system_prompt else []
messages.append(
    {"role": "user", "content": "What is the capital of Australia? Answer in one word."}
)

chat_response = requests.post(
    f"http://localhost:{port}/v1/chat/completions",
    json={"model": model_id, "messages": messages, "max_tokens": 64},
    timeout=60,
)
chat_response.raise_for_status()
print(chat_response.json()["choices"][0]["message"]["content"])

If you get a sensible answer (e.g. *Canberra*), the NIM is deployed and working.

> **Note:** Nemotron Nano is a *reasoning* model and may include a brief
> chain-of-thought before its final answer. That is normal and is handled fine
> by the evaluation cells in the next notebook.

<br><hr>

## **Managing the NIM**

Useful commands (run from a terminal or prefix with `!` in a cell):

```bash
# Substitute your own $NIM_CONTAINER for 'nemotron-nano' below.
docker ps                     # confirm 'nemotron-nano' is Up
docker logs nemotron-nano     # inspect startup / errors
docker stop nemotron-nano     # stop serving (frees the GPU)
docker start nemotron-nano    # resume (fast — cache is warm)
docker rm -f nemotron-nano    # remove the container
```

The model weights persist in `~/.cache/nim`, so removing and recreating the
container does **not** re-download the model.

<br><hr>

**Next:** proceed to
[`01-NIM-Evaluation.ipynb`](01-NIM-Evaluation.ipynb) to evaluate this NIM. Its
configuration cell already points at `http://localhost:8000` and the served
model id `nvidia/nemotron-nano-9b-v2`.